<a href="https://colab.research.google.com/github/nayansc722-stack/OT-Security-Projects/blob/Project-2-OT-Network-Security-Analysis-Tool/Project_2_Network_Security_Analysis_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.chdir('/content/drive/MyDrive/Colab Notebooks/ot_inspect_tool/ot_inspect')
os.listdir('.')

['README.md', 'ot_inspect.py', 'config.yaml', 'reports', 'core']

In [ ]:
# Open and print the current config so you can see it
with open('config.yaml', 'r') as f:
    print(f.read())

# ═══════════════════════════════════════════════════════════════════════════════
# ot-inspect configuration file
#
# Edit this file once for your network. The tool reads it at startup.
# No code changes needed when moving between environments.
#
# Usage:
#   python ot_inspect.py --pcap capture.pcap --config config.yaml
# ═══════════════════════════════════════════════════════════════════════════════

network:
  # IPs authorised to send commands to PLCs (SCADA masters, HMIs, engineering WS)
  # All other sources sending write/control commands will trigger alerts
  authorised_masters:
    - "10.10.10.20"       # Engineering Workstation (4SICS lab)
    # - "192.168.10.100"  # Add your real master IPs here

  # Known PLC/RTU IP addresses
  # Used to detect when a field device initiates unexpected outbound connections
  plc_addresses:
    - "10.10.10.10"       # Siemens S7 PLC (4SICS lab)
    # - "192.168.1.10"    # Add your real PLC IPs here

  # IT network prefixes — OT protocols should 

In [11]:
import os

path = "/content/drive/MyDrive/Colab Notebooks/ot_inspect_tool/ot_inspect"
print("Folder exists:", os.path.exists(path))
print()
print("Contents:")
for f in os.listdir(path):
    print(" ", f)

Folder exists: True

Contents:
  README.md
  ot_inspect.py
  config.yaml
  reports
  core


In [13]:
report_gen_path = "/content/drive/MyDrive/Colab Notebooks/ot_inspect_tool/ot_inspect/reports/report_generator.py"

with open(report_gen_path, 'r') as f:
    content = f.read()

# Fix: always wrap axes in a list so len() works
content = content.replace(
    'if not isinstance(axes, (list, type(plt.subplots(1,1)[1]))):\n        axes = [axes]',
    'if not hasattr(axes, "__len__"):\n        axes = [axes]'
)

with open(report_gen_path, 'w') as f:
    f.write(content)

print("✅ Fixed")

✅ Fixed


In [14]:
import subprocess

PCAP = "/content/drive/MyDrive/data/4SICS-GeekLounge-151020.pcap"

result = subprocess.run(
    ["python", "ot_inspect.py",
     "--pcap",   PCAP,
     "--config", "config.yaml",
     "--out",    "/content/drive/MyDrive/Colab Notebooks/reports/"],
    capture_output=True,
    text=True,
    cwd="/content/drive/MyDrive/Colab Notebooks/ot_inspect_tool/ot_inspect"
)

print(result.stdout)
if result.stderr:
    print("ERRORS:", result.stderr[:2000])


  ╔═══════════════════════════════════════════╗
  ║         ot-inspect  v1.0                  ║
  ║   OT Network Security Analysis Tool       ║
  ║   Protocols: Modbus TCP · S7comm          ║
  ╚═══════════════════════════════════════════╝

[config] Loaded: config.yaml

  ──────────────────────────────────────────────────
  STEP 1: Protocol Discovery
  ──────────────────────────────────────────────────
[discovery] Loading: /content/drive/MyDrive/data/4SICS-GeekLounge-151020.pcap
[discovery] Total packets: 246,137
[discovery] Scanning for ICS protocols...

  PROTOCOL DISCOVERY RESULTS
  Total packets    : 246,137
  IP packets       : 239,267
  TCP packets      : 208,940
  UDP packets      : 27,587
  Unique hosts     : 12

  ICS protocols found: 1

  ✅ S7comm (Siemens) (port 102)
     Packets  : 208,940
     Sources  : 3 unique IPs
     Targets  : 3 unique IPs


  ──────────────────────────────────────────────────
  STEP 2: Protocol Parsing
  ────────────────────────────────────────────